In [3]:
# =========================================================
# 0. IMPORT LIBRARIES
# =========================================================
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import joblib

from itertools import product

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from catboost import CatBoostRegressor
import statsmodels.api as sm

In [4]:
# =========================================================
# 1. LOAD + CLEAN DATA
# Keep logic close to OLS baseline
# =========================================================
file_path = r"C:\Users\Jing Xuan\Desktop\Y3S2\DSE3101\DSE3101 Group Project\hdb_with_amenities_macro.csv"
df_raw = pd.read_csv(file_path)

print(f"Initial shape: {df_raw.shape}")

# Drop rows with missing values
df = df_raw.dropna().copy()
print(f"After dropping nulls: {df.shape} (dropped {len(df_raw) - len(df)})")

# Drop rare flat types
mask_flat = ~df["flat_type"].isin(["1 ROOM", "MULTI-GENERATION"])
n_before = len(df)
df = df[mask_flat].copy()
print(f"After dropping 1 ROOM and MULTI-GENERATION: {df.shape} (dropped {n_before - len(df)})")

# Drop 2020
df["year_temp"] = pd.to_datetime(df["month"]).dt.year
n_before = len(df)
df = df[df["year_temp"] != 2020].drop(columns="year_temp").copy()
print(f"After dropping 2020: {df.shape} (dropped {n_before - len(df)})")

# Log target
df["log_resale_price_real"] = np.log(df["resale_price_real"])

# num_primary_1km from pipe-separated school names
df["num_primary_1km"] = df["primary_schools_1km"].apply(
    lambda x: len(x.split("|")) if pd.notna(x) and x != "" else 0
)

print(f"num_primary_1km — mean: {df['num_primary_1km'].mean():.2f}, max: {df['num_primary_1km'].max()}")

Initial shape: (157821, 37)
After dropping nulls: (114147, 37) (dropped 43674)
After dropping 1 ROOM and MULTI-GENERATION: (114057, 37) (dropped 90)
After dropping 2020: (96796, 37) (dropped 17261)
num_primary_1km — mean: 2.99, max: 9


In [5]:
# =========================================================
# 2. FEATURE ENGINEERING
# =========================================================
def parse_remaining_lease(s):
    match = re.match(r"(\d+) years?(?: (\d+) months?)?", str(s))
    if match:
        years = int(match.group(1))
        months = int(match.group(2)) if match.group(2) else 0
        return round(years + months / 12, 2)
    return np.nan

df["remaining_lease_years"] = df["remaining_lease"].apply(parse_remaining_lease)

# Extract lower floor from storey_range
df["floor_lower"] = df["storey_range"].str.extract(r"^(\d+)").astype(int)

df["floor_category"] = pd.cut(
    df["floor_lower"],
    bins=[0, 5, 12, float("inf")],
    labels=["Low", "Mid", "High"],
    right=True
)

# Numeric code version for convenience if needed
floor_map = {"Low": 0, "Mid": 1, "High": 2}
df["floor_category_code"] = df["floor_category"].map(floor_map).astype(int)

# Time variable for splitting
df["year"] = pd.to_datetime(df["month"]).dt.year

target = "log_resale_price_real"

print(df[[
    "remaining_lease", "remaining_lease_years",
    "storey_range", "floor_lower",
    "floor_category", "year"
]].head(10))

          remaining_lease  remaining_lease_years storey_range  floor_lower  \
23333   64 years 01 month                  64.08     01 TO 03            1   
23334   64 years 01 month                  64.08     07 TO 09            7   
23335            59 years                  59.00     04 TO 06            4   
23336  58 years 02 months                  58.17     04 TO 06            4   
23337   58 years 01 month                  58.08     01 TO 03            1   
23338  64 years 02 months                  64.17     07 TO 09            7   
23339  58 years 02 months                  58.17     04 TO 06            4   
23340   59 years 01 month                  59.08     04 TO 06            4   
23341            64 years                  64.00     07 TO 09            7   
23342  54 years 04 months                  54.33     04 TO 06            4   

      floor_category  year  
23333            Low  2021  
23334            Mid  2021  
23335            Low  2021  
23336            Low  202

In [6]:
# =========================================================
# 3. TIME-AWARE TRAIN / VALIDATION / TEST SPLIT
# Example:
# train = 2021-2022
# val   = 2023
# test  = 2024-2025
# Adjust if needed
# =========================================================
train_df = df[df["year"] <= 2022].copy()
val_df   = df[df["year"] == 2023].copy()
test_df  = df[df["year"] >= 2024].copy()

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain years:", sorted(train_df["year"].unique()))
print("Validation years:", sorted(val_df["year"].unique()))
print("Test years:", sorted(test_df["year"].unique()))

Train shape: (40512, 44)
Validation shape: (18174, 44)
Test shape: (38110, 44)

Train years: [2021, 2022]
Validation years: [2023]
Test years: [2024, 2025]


In [7]:
# =========================================================
# 4. FEATURE SET
# Keep it aligned with OLS baseline for fair comparison
# OLS excluded floor_area_sqm and dist_cbd_m
# =========================================================
continuous_features = [
    "remaining_lease_years",
    "nearest_train_dist_m",
    "dist_nearest_hawker_m",
    "dist_nearest_primary_m",
    "num_primary_1km",
    "dist_nearest_park_m",
    "dist_nearest_sportsg_m",
    "dist_nearest_mall_m",
    "dist_nearest_healthcare_m"
]

categorical_features = ["flat_type", "town", "floor_category"]

feature_cols = continuous_features + categorical_features

X_train = train_df[feature_cols].copy()
X_val   = val_df[feature_cols].copy()
X_test  = test_df[feature_cols].copy()

y_train = train_df[target].copy()
y_val   = val_df[target].copy()
y_test  = test_df[target].copy()

y_train_actual = train_df["resale_price_real"].copy()
y_val_actual   = val_df["resale_price_real"].copy()
y_test_actual  = test_df["resale_price_real"].copy()

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

X_train shape: (40512, 12)
X_val shape: (18174, 12)
X_test shape: (38110, 12)


In [8]:
# =========================================================
# 5. EVALUATION FUNCTIONS
# =========================================================
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.array(y_true) - np.array(y_pred)) ** 2))

def linlin_loss(y_true, y_pred, underpredict_weight=2.0):
    """
    Asymmetric linear loss penalising underprediction more heavily than overprediction.
    """
    errors = np.array(y_true) - np.array(y_pred)  # positive = underprediction
    loss = np.where(errors > 0, underpredict_weight * errors, -errors)
    return np.mean(loss)

def evaluate_predictions(y_true_actual, y_pred_actual, underpredict_weight=2.0):
    return {
        "RMSE": rmse(y_true_actual, y_pred_actual),
        "R2": r2_score(y_true_actual, y_pred_actual),
        "LinLin": linlin_loss(y_true_actual, y_pred_actual, underpredict_weight=underpredict_weight)
    }

In [ ]:
# =========================================================
# 6. CATBOOST HYPERPARAMETER TUNING
# Tune on validation set only
# =========================================================
cat_feature_indices = [X_train.columns.get_loc(col) for col in categorical_features]

param_grid = {
    "depth": [6, 8, 10],
    "learning_rate": [0.01, 0.03, 0.1],
    "l2_leaf_reg": [3, 5, 7],
    "iterations": [1000, 2000]
}

tuning_results = []

for depth, lr, l2, n_iter in product(
    param_grid["depth"],
    param_grid["learning_rate"],
    param_grid["l2_leaf_reg"],
    param_grid["iterations"]
):
    model = CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="RMSE",
        depth=depth,
        learning_rate=lr,
        l2_leaf_reg=l2,
        iterations=n_iter,
        random_seed=42,
        early_stopping_rounds=100,
        verbose=0
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_feature_indices,
        eval_set=(X_val, y_val),
        use_best_model=True
    )

    y_val_pred_log_cb = model.predict(X_val)
    y_val_pred_cb = np.exp(y_val_pred_log_cb)

    metrics = evaluate_predictions(y_val_actual, y_val_pred_cb)

    tuning_results.append({
        "depth": depth,
        "learning_rate": lr,
        "l2_leaf_reg": l2,
        "iterations": n_iter,
        "best_iteration": model.get_best_iteration(),
        "RMSE": metrics["RMSE"],
        "R2": metrics["R2"],
        "LinLin": metrics["LinLin"]
    })

tuning_results_df = pd.DataFrame(tuning_results).sort_values(["RMSE", "LinLin"])
print(tuning_results_df.head(10))